In [ ]:
# ── Notebook parameters ─────────────────────────────────────────
# Defaults for interactive use. extract_notebook_figures.py will
# override these via a cell injected right below this one.
#
# Add notebook-level globals here, e.g.:
INSET_COL    = 0       # which column of the mosaic to inspect (0..N_TOP-1)
INSET_LEFT   = 1/5.
INSET_RIGHT  = 1/2. + 0.1
INSET_DOWN   = 1/5.
INSET_UP     = 1/2. + 0.1
INSET_TARGET = 64     # target resolution inside the inset box


# Two-Moons NTK Eigenfunction Visualization

This notebook studies the empirical Neural Tangent Kernel (NTK) along the
training trajectory of a binary classifier on the two-moons dataset.

The NTK at a fixed parameter vector $\theta$ is

$$K(x, y) = \bigl\langle \nabla_\theta f(x; \theta),\ \nabla_\theta f(y; \theta) \bigr\rangle = J(\theta)\, J(\theta)^\top$$

where $f$ is the **pre-sigmoid logit** of the classifier. We train a small
NTK-parameterised MLP, save a user-chosen set of checkpoints across training,
and visualise the top-$K$ NTK eigenfunctions at each checkpoint via two
complementary methods:

1. **Direct eigendecomposition.** Build $K$ on a regular grid and call
   `np.linalg.eigh`. Cheap, model-agnostic, but the eigenfunctions only live
   on the grid points.
2. **Function-space diagonalisation via `NeuralIsometry`.** Train an
   `EulerianIsometry` with `reg_cob.py` per checkpoint so that
   $\{Q\phi_k\}_k$ approximately diagonalises the NTK; eigenvalues are then
   $\lambda_k = \|\nabla_\theta \langle Q^* f_\theta,\, \phi_k\rangle\|^2$.

The final figure is a $K \times M$ grid where $M$ is the number of loaded
isometries (one per classifier checkpoint) and $K$ is the number of top
eigenfunctions: each column is one checkpoint along training, and the rows
are the top-$K$ NTK eigenfunctions ordered by descending eigenvalue.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')

import os
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from infidictionary.networks import NTKMLPNeuralField
from infidictionary.ntk import estimate_ntk

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## 1. Two-moons dataset

We generate two interleaving moons and rescale them into $[0, 1]^2$ so the
classifier's input domain matches the Fourier dictionary used later by the
isometry. The classifier output is a single **logit** (no sigmoid), large
positive on one moon, large negative on the other.


In [ ]:
from sklearn.datasets import make_moons

N_TRAIN  = 2000
NOISE    = 0.10
SEED     = 0

X_raw, y_np = make_moons(n_samples=N_TRAIN, noise=NOISE, random_state=SEED)

# Rescale into [0.05, 0.95]^2 so points stay clear of the unit-square edges.
xmin = X_raw.min(axis=0)
xmax = X_raw.max(axis=0)
X_np = 0.05 + 0.90 * (X_raw - xmin) / (xmax - xmin)

X_train = torch.tensor(X_np,   dtype=torch.float32, device=device)             # (N, 2)
y_train = torch.tensor(y_np,   dtype=torch.float32, device=device).unsqueeze(-1)  # (N, 1) in {0,1}

print(f'X_train: {tuple(X_train.shape)}  range=[{X_np.min():.3f}, {X_np.max():.3f}]')
print(f'class balance: {int(y_np.sum())} positives / {N_TRAIN}')

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(X_np[y_np == 0, 0], X_np[y_np == 0, 1], s=8, c='tab:blue', label='class 0')
ax.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], s=8, c='tab:red',  label='class 1')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
ax.set_title('Two-moons dataset (rescaled to $[0,1]^2$)')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()


## 2. Train the classifier and save checkpoints

The classifier is an `NTKMLPNeuralField` with three hidden layers. It outputs
a single scalar logit; we minimise binary cross-entropy with logits.

The set of training steps to checkpoint is controlled by the
`CHECKPOINT_STEPS` list — edit it to whatever schedule you want. Each
chosen step is saved to
`outputs/ntk/two_moons_step_<step>.pt`. Re-running the cell skips training
when all checkpoints are already on disk.


In [ ]:
CHECKPOINT_DIR = '../outputs/ntk'
LR             = 1e-2
BATCH_SIZE     = 256

# Edit this list to control which steps get a checkpoint. Densely sample
# the early steps (where the NTK changes fastest) and sparser later. The
# total number of training steps is taken to be max(CHECKPOINT_STEPS).
CHECKPOINT_STEPS = [1, 250, 500, 5000]

# Architecture used by both the classifier and the NTK regularizer config.
MODEL_KWARGS = dict(
    input_dim=2,
    output_dim=1,
    hidden_dims={'a': 256, 'b': 256, 'c': 256},
    n_fourier_features=0,
    fourier_sigma=0.0,
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CHECKPOINT_STEPS = sorted(set(int(s) for s in CHECKPOINT_STEPS))
assert CHECKPOINT_STEPS, 'CHECKPOINT_STEPS must contain at least one step.'
assert all(s >= 1 for s in CHECKPOINT_STEPS), 'all checkpoint steps must be >= 1.'

NUM_CHECKPOINTS = len(CHECKPOINT_STEPS)
N_STEPS         = CHECKPOINT_STEPS[-1]   # train just long enough to hit the last checkpoint

# Backwards-compatible aliases used by the rest of the notebook.
checkpoint_steps = CHECKPOINT_STEPS

def ckpt_path(step: int) -> str:
    return os.path.join(CHECKPOINT_DIR, f'two_moons_step_{step:04d}.pt')

CHECKPOINT_PATHS = [ckpt_path(s) for s in checkpoint_steps]
print(f'Will train for {N_STEPS} steps; checkpointing at: {checkpoint_steps}')


In [ ]:
def train_two_moons_classifier():
    torch.manual_seed(SEED)
    model = NTKMLPNeuralField(**MODEL_KWARGS).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=LR)
    bce   = torch.nn.BCEWithLogitsLoss()

    pending = set(checkpoint_steps)
    losses  = []

    pbar = tqdm(range(1, N_STEPS + 1), desc='Training two-moons classifier')
    for step in pbar:
        idx = torch.randint(0, N_TRAIN, (BATCH_SIZE,), device=device)
        logits = model(X_train[idx])
        loss   = bce(logits, y_train[idx])

        optim.zero_grad()
        loss.backward()
        optim.step()
        losses.append(loss.item())

        if step in pending:
            torch.save({
                'model_state_dict': model.state_dict(),
                'model_kwargs':     MODEL_KWARGS,
                'step':             step,
                'loss':             loss.item(),
            }, ckpt_path(step))
            pending.discard(step)
            pbar.set_postfix({'step': step, 'loss': f'{loss.item():.4f}'})

    return losses

if all(os.path.exists(p) for p in CHECKPOINT_PATHS):
    print(f'All {NUM_CHECKPOINTS} checkpoints already exist in {CHECKPOINT_DIR}; skipping training.')
    losses = []
else:
    losses = train_two_moons_classifier()

if losses:
    plt.figure(figsize=(7, 3))
    plt.semilogy(losses)
    plt.xlabel('step'); plt.ylabel('BCE loss'); plt.grid(True, alpha=0.4)
    plt.title('Two-moons classifier training loss'); plt.tight_layout(); plt.show()


In [ ]:
# Quick sanity check: load the final checkpoint and visualise the decision
# surface (the logit). Large positive ⇒ class 1, large negative ⇒ class 0.
model = NTKMLPNeuralField(**MODEL_KWARGS).to(device)
for idx in range(len(CHECKPOINT_PATHS)):
    ckpt = torch.load(CHECKPOINT_PATHS[idx], map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    N_VIS = 100
    xs = torch.linspace(0.0, 1.0, N_VIS, device=device)
    gx, gy = torch.meshgrid(xs, xs, indexing='xy')
    grid_coords = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=-1)
    with torch.no_grad():
        logits_grid = model(grid_coords).reshape(N_VIS, N_VIS).cpu().numpy()

    vmax = float(np.abs(logits_grid).max())
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(logits_grid, extent=(0, 1, 0, 1), origin='lower',
                cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.scatter(X_np[y_np == 0, 0], X_np[y_np == 0, 1], s=4, c='blue',   alpha=0.5)
    ax.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], s=4, c='red', alpha=0.5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    ax.set_title(f'Logit f(x) at step {ckpt["step"]}')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()


## 3. Direct NTK eigendecomposition (baseline)

For each checkpoint we evaluate the NTK matrix on a regular grid using
`estimate_ntk`, diagonalise it with `np.linalg.eigh`, and plot the top
eigenfunctions.

The NTK matrix scales as $(N\,C)^2$ in memory and the per-checkpoint cost
also scales with the number of model parameters, so keep `NTK_GRID_RES`
small. For a 256-wide 3-layer NTK MLP, a 22×22 grid (484 points) keeps the
Jacobian under ~1 GB.


In [ ]:
NTK_GRID_RES = 16                # NTK_GRID_RES^2 grid points
N_EIG_BASELINE = 4              # top-K eigenfunctions to plot

ntk_xs = torch.linspace(0.0, 1.0, NTK_GRID_RES, device=device)
ntk_gx, ntk_gy = torch.meshgrid(ntk_xs, ntk_xs, indexing='xy')
ntk_coords = torch.stack([ntk_gx.reshape(-1), ntk_gy.reshape(-1)], dim=-1)
N_grid = ntk_coords.shape[0]
print(f'NTK grid: {NTK_GRID_RES}x{NTK_GRID_RES} = {N_grid} points')


In [ ]:
# For each checkpoint, compute K and its top eigenvalues / eigenvectors.
baseline_results = []   # list of dicts: {step, evals (N_grid,), evecs (N_grid, N_EIG)}
model_tmp = NTKMLPNeuralField(**MODEL_KWARGS).to(device)

for path, step in zip(CHECKPOINT_PATHS, checkpoint_steps):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model_tmp.load_state_dict(ckpt['model_state_dict'])
    model_tmp.eval()

    K = estimate_ntk(model_tmp, ntk_coords, batch_size=8)              # (N, N), output_dim=1
    K_np = K.cpu().numpy()

    evals_raw, evecs = np.linalg.eigh(K_np)              # ascending; evecs are Euclidean unit-norm
    order = np.argsort(evals_raw)[::-1]
    evals = evals_raw[order] / N_grid                    # L^2 scale (quadrature weight 1/N)
    evecs = evecs[:, order]                              # (N_grid, N_grid)

    # np.linalg.eigh returns Euclidean unit-norm vectors (sum v_i^2 = 1).
    # Our L^2 quadrature approximates <f,g> ≈ (1/N) sum f_i g_i, so the
    # L^2-normalized eigenfunction is sqrt(N) * v.  Rescale so that the
    # displayed scale matches the function-space eigenfunctions (which are
    # L^2-normalized Fourier atoms pushed through the isometry).
    evecs = evecs * np.sqrt(N_grid)

    baseline_results.append({
        'step':  step,
        'evals': evals[:N_EIG_BASELINE],
        'evecs': evecs[:, :N_EIG_BASELINE],
    })
    last = N_EIG_BASELINE - 1
    ratio = evals[0] / evals[last] if evals[last] > 0 else float('inf')
    print(f'step={step:5d}  ||K||_F = {np.linalg.norm(K_np):.3e}  '
          f'top eig = {evals[0]:.3e}  ratio λ_0/λ_{last} = {ratio:.2f}')

In [ ]:
NTK_GRID_HIGH_RES  = 32   # per-side resolution; total points = NTK_GRID_HIGH_RES^2
NTK_SAVE_BATCH_SIZE = 32    # number of grid points per Jacobian call; lower = less VRAM

ntk_hi_xs = torch.linspace(0.0, 1.0, NTK_GRID_HIGH_RES, device=device)
ntk_hi_gx, ntk_hi_gy = torch.meshgrid(ntk_hi_xs, ntk_hi_xs, indexing='xy')
ntk_hi_coords = torch.stack([ntk_hi_gx.reshape(-1), ntk_hi_gy.reshape(-1)], dim=-1)
ntk_hi_coords_np = ntk_hi_coords.cpu().numpy()   # (N_hi, 2)
N_hi = ntk_hi_coords.shape[0]
print(f'Dense NTK grid: {NTK_GRID_HIGH_RES}x{NTK_GRID_HIGH_RES} = {N_hi} points  '
      f'(Jacobian batch size: {NTK_SAVE_BATCH_SIZE})')

model_hi = NTKMLPNeuralField(**MODEL_KWARGS).to(device)

for path, step in tqdm(list(zip(CHECKPOINT_PATHS, checkpoint_steps)),
                        desc='Saving dense kernels'):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model_hi.load_state_dict(ckpt['model_state_dict'])
    model_hi.eval()

    K_hi = estimate_ntk(model_hi, ntk_hi_coords, batch_size=NTK_SAVE_BATCH_SIZE)
    K_hi_np = K_hi.cpu().numpy()

    kernel_path = os.path.join(CHECKPOINT_DIR, f'kernel_at_step_{step:04d}.npy')
    points_path = os.path.join(CHECKPOINT_DIR, f'points_at_step_{step:04d}.npy')
    np.save(kernel_path, K_hi_np)
    np.save(points_path, ntk_hi_coords_np)
    print(f'step={step:5d}  kernel {K_hi_np.shape} → {kernel_path}')

print(f'\nAlso saved points {ntk_hi_coords_np.shape} → {points_path}')

In [ ]:
# Plot top-K eigenfunctions for each checkpoint as a (NUM_CHECKPOINTS x N_EIG_BASELINE)
# mosaic. Columns = eigenfunction index (descending λ), rows = training step.
# A narrow colorbar column on the right uses the shared per-row scale from row_vmaxes
# (falls back to per-image scale if row_vmaxes has not been computed yet).
try:
    _vmaxes = row_vmaxes
except NameError:
    _vmaxes = {}

fig, axes = plt.subplots(
    NUM_CHECKPOINTS, N_EIG_BASELINE + 1,
    figsize=(N_EIG_BASELINE * 1.2 + 0.4, NUM_CHECKPOINTS * 1.2),
    gridspec_kw={'wspace': 0.02, 'hspace': 0.05,
                 'width_ratios': [1] * N_EIG_BASELINE + [0.05]},
    squeeze=False,
)

# Capture the vmax used per row so the inset below uses the exact same scale.
_used_vmax = {}
for r, res in enumerate(baseline_results):
    vmax = _vmaxes.get(res['step'], float(np.abs(res['evecs']).max()) + 1e-12)
    _used_vmax[res['step']] = vmax
    im = None
    for c in range(N_EIG_BASELINE):
        ax = axes[r, c]
        ax.set_xticks([]); ax.set_yticks([])
        v = res['evecs'][:, c]
        v_img = v.reshape(NTK_GRID_RES, NTK_GRID_RES)
        im = ax.imshow(v_img, extent=(0, 1, 0, 1), origin='lower',
                  cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_aspect('equal')
        ax.text(0.03, 0.03, f'λ={res["evals"][c]:.1e}',
                transform=ax.transAxes, fontsize=5, va='bottom', ha='left',
                bbox=dict(facecolor='white', alpha=0.6, pad=1, edgecolor='none'))
        if r == 0:
            ax.set_title(f'#{c}', fontsize=8)
        if c == 0:
            ax.set_ylabel(f'step {res["step"]}', fontsize=8)
    fig.colorbar(im, cax=axes[r, N_EIG_BASELINE])

fig.suptitle(
    f'Direct NTK eigenfunctions on a {NTK_GRID_RES}×{NTK_GRID_RES} grid\n'
    f'rows = checkpoints,  columns = top-{N_EIG_BASELINE} eigenfunctions (descending λ)',
    fontsize=10,
)
plt.tight_layout(); plt.show()

# ── Inset zoom on one eigenfunction column ───────────────────────────────────
import matplotlib.patches as mpatches

x_lo = int(round(INSET_LEFT  * NTK_GRID_RES))
x_hi = max(x_lo + 1, int(round(INSET_RIGHT * NTK_GRID_RES)))
y_lo = int(round(INSET_DOWN  * NTK_GRID_RES))
y_hi = max(y_lo + 1, int(round(INSET_UP    * NTK_GRID_RES)))
inset_h, inset_w = y_hi - y_lo, x_hi - x_lo

fig, axes = plt.subplots(
    NUM_CHECKPOINTS, 2,
    figsize=(2 * 1.8, NUM_CHECKPOINTS * 1.8),
    gridspec_kw={'wspace': 0.05, 'hspace': 0.05},
    squeeze=False,
)
for r, res in enumerate(baseline_results):
    v_img = res['evecs'][:, INSET_COL].reshape(NTK_GRID_RES, NTK_GRID_RES)
    crop  = v_img[y_lo:y_hi, x_lo:x_hi]
    vmax  = _used_vmax[res['step']]   # same scale as the main mosaic above

    # Left: full eigenfunction with the inset box overlaid.
    ax_full = axes[r, 0]
    ax_full.imshow(v_img, extent=(0, 1, 0, 1), origin='lower',
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   interpolation='nearest')
    ax_full.add_patch(mpatches.Rectangle(
        (INSET_LEFT, INSET_DOWN),
        INSET_RIGHT - INSET_LEFT, INSET_UP - INSET_DOWN,
        linewidth=1.5, edgecolor='red', facecolor='none',
    ))
    ax_full.set_xticks([]); ax_full.set_yticks([]); ax_full.set_aspect('equal')
    ax_full.set_ylabel(f'step {res["step"]}', fontsize=8)
    if r == 0:
        ax_full.set_title(f'#{INSET_COL}  full  ({NTK_GRID_RES}×{NTK_GRID_RES})', fontsize=8)

    # Right: cropped inset, framed in red.
    ax_crop = axes[r, 1]
    ax_crop.imshow(crop, extent=(INSET_LEFT, INSET_RIGHT, INSET_DOWN, INSET_UP),
                   origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   interpolation='nearest')
    for sp in ax_crop.spines.values():
        sp.set_edgecolor('red'); sp.set_linewidth(1.5)
    ax_crop.set_xticks([]); ax_crop.set_yticks([]); ax_crop.set_aspect('equal')
    if r == 0:
        ax_crop.set_title(f'inset  ({inset_w}×{inset_h} px)', fontsize=8)

fig.suptitle(
    f'Inset on eigenfunction #{INSET_COL} — direct grid solver  '
    f'(grid is {NTK_GRID_RES}×{NTK_GRID_RES})',
    fontsize=10,
)
plt.tight_layout(); plt.show()

## 4. Function-space NTK eigenfunctions via `NeuralIsometry`

For every saved classifier checkpoint, run `reg_cob.py` once with the
`two_moons` NTK experiment to learn an `EulerianIsometry` $Q$ that
diagonalises the NTK at that checkpoint:

```bash
for ckpt in outputs/ntk/two_moons_step_*.pt; do
    python reg_cob.py +ntk_experiment=two_moons \
        regularizer.ntk_model_weights_path="$ckpt" \
        n_epochs=2000
done
```

Each run writes its own `outputs/checkpoints/wandb-<id>/` (or timestamp)
directory. Collect the directory names below — one per training step,
ordered to match `checkpoint_steps`.


In [ ]:
# Map a two-moons checkpoint step (must be in `checkpoint_steps`) to the
# NeuralIsometry checkpoint directory produced by reg_cob.py for that step.
# You only need to fill in the steps you actually trained an isometry for —
# the rest are silently skipped, and the final grid shrinks accordingly.

ISOMETRY_CKPT_BY_STEP = {
    # checkpoint_steps[i]: '../outputs/checkpoints/wandb-xxxxxxxx',
    # e.g.
    1: ('../outputs/checkpoints/wandb-7e7zbmtv', 'step_2000.pt'),
    250: ('../outputs/checkpoints/wandb-58a0o1ip', 'step_2000.pt'),
    500:  ('../outputs/checkpoints/wandb-pakhvui3', 'step_2000.pt'),
    5000: ('../outputs/checkpoints/wandb-wfxt9ioq', 'step_2000.pt'),
}

# Validate keys.
unknown = set(ISOMETRY_CKPT_BY_STEP) - set(checkpoint_steps)
if unknown:
    raise ValueError(
        f'ISOMETRY_CKPT_BY_STEP contains steps not in checkpoint_steps: {sorted(unknown)}.\n'
        f'Valid steps: {checkpoint_steps}'
    )

if not ISOMETRY_CKPT_BY_STEP:

    print(f'ISOMETRY_CKPT_BY_STEP is empty. Run reg_cob.py for at least one '
          f'classifier checkpoint and add the resulting directory above before '
          f'running the next cells.')
else:
    print(f'Will load {len(ISOMETRY_CKPT_BY_STEP)} of {NUM_CHECKPOINTS} '
          f'isometries:  steps={sorted(ISOMETRY_CKPT_BY_STEP)}')


In [ ]:
import yaml
from omegaconf import OmegaConf
from hydra.utils import instantiate
from infidictionary.neural_isometries import NeuralIsometry
from infidictionary.dictionaries import InfiDictionary

isometry_bundles = {}   # step -> dict with isometry, dictionary, kwargs

for step, tup in sorted(ISOMETRY_CKPT_BY_STEP.items()):
    ckpt_dir, ckpt_file = tup
    with open(f'{ckpt_dir}/config.yaml') as f:
        conf = OmegaConf.create(yaml.safe_load(f))
    ckpt_dir = f"{ckpt_dir}/{ckpt_file}"
    iso: NeuralIsometry = instantiate(conf.neural_isometry).to(device)
    dictionary: InfiDictionary = instantiate(conf.initial_dictionary)

    ckpt = torch.load(f'{ckpt_dir}',
                      map_location=device, weights_only=False)
    iso.load_state_dict(ckpt['models']['neural_isometry'])

    msk = OmegaConf.to_container(conf.get('model_state_kwargs', {'num_steps': 200}), resolve=True)
    pfk = OmegaConf.to_container(conf.get('pushforward_kwargs',
                                          {'start_time': 0.0, 'end_time': 1.0}), resolve=True)

    iso.shuffle_model_state(**msk)
    iso.eval()

    isometry_bundles[step] = dict(
        isometry=iso,
        dictionary=dictionary,
        model_state_kwargs=msk,
        pushforward_kwargs=pfk,
        ckpt_dir=ckpt_dir,
        epoch=ckpt.get('epoch', None),
    )
    print(f'Loaded {ckpt_dir}  (step {step}, epoch {ckpt.get("epoch", "?")})')

print(f'\nLoaded {len(isometry_bundles)} isometries.')


## 5. Top-$K$ NTK eigenfunctions per checkpoint

For each (classifier checkpoint, isometry) pair we evaluate, for each atom
$\phi_a$ in a truncated Fourier sub-dictionary,

$$\lambda_a = \bigl\|\nabla_\theta \langle Q^* f_\theta,\, \phi_a \rangle_{L^2(\mathrm{src})}\bigr\|^2$$

then sort atoms by $\lambda_a$ (descending) and render the corresponding
pushed-forward atom $Q\phi_a$ on a regular target-domain grid. The result
is a $K \times M$ mosaic where $M$ is the number of loaded isometries —
one column per checkpoint, top-$K$ eigenfunctions per column ($K$ =
`N_TOP`).

`compute_ntk_qf` mirrors the `NTKRegularizer` energy: it evaluates
$\langle Q^* f_\theta, \phi_a \rangle$ as a function of the NTK model
parameters and squares the parameter-Jacobian.


In [ ]:
from torch.func import functional_call
from infidictionary.utils import pairwise_inner_product
from infidictionary.domain_samplers import SquareSampler

NUM_TRUNCATED_QF = 10        # atom truncation for QF computation
N_TOP            = 4        # eigenfunctions per checkpoint
N_VIS_ATOM       = 60        # tile resolution for pushforward visualisation

def compute_ntk_qf(ntk_model, isometry, dictionary, indices,
                   tgt_coords, tgt_logabsdet, pushforward_kwargs):
    '''Per-atom NTK quadratic form  lambda_a = ||grad_theta <Q* f_theta, phi_a>||^2.'''
    with torch.no_grad():
        src_coords, src_logabsdet, _ = isometry.pullback(
            tgt_coords=tgt_coords,
            tgt_logabsdet=tgt_logabsdet,
            tgt_field=torch.zeros(1, tgt_coords.shape[0], 1, device=tgt_coords.device),
            **pushforward_kwargs,
        )
    src_coords = src_coords.detach()
    src_logabsdet = src_logabsdet.detach()
    phi_src = dictionary.get_atoms(src_coords, indices)              # (A, N, C)

    param_names = [name for name, _ in ntk_model.named_parameters()]

    def c_from_params(*params):
        ntk_dict = dict(zip(param_names, params))
        f_vals = functional_call(ntk_model, ntk_dict, (tgt_coords,))
        if f_vals.dim() == 1:
            f_vals = f_vals.unsqueeze(-1)
        _, _, qsf = isometry.pullback(
            tgt_coords=tgt_coords,
            tgt_logabsdet=tgt_logabsdet,
            tgt_field=f_vals.unsqueeze(0),
            **pushforward_kwargs,
        )
        return pairwise_inner_product(qsf, phi_src, src_logabsdet).squeeze(0)

    J_tuple = torch.autograd.functional.jacobian(
        c_from_params,
        tuple(ntk_model.parameters()),
        create_graph=False,
        vectorize=True,
    )
    A = indices.shape[0]
    return sum(j.reshape(A, -1).pow(2).sum(-1) for j in J_tuple).detach()


In [ ]:
# Pre-compute the visualisation grid (target-domain coords).
vis_coords = SquareSampler(stratified=True, add_noise=False).sample(N_VIS_ATOM).to(device)
vis_logabsdet = torch.zeros(vis_coords.shape[0], device=device)

# QF evaluation grid — coarser, since we re-run a Jacobian per (checkpoint, atom set).
qf_coords  = SquareSampler(stratified=True, add_noise=False).sample(NTK_GRID_RES).to(device)
qf_logabsdet = torch.zeros(qf_coords.shape[0], device=device)

ntk_model = NTKMLPNeuralField(**MODEL_KWARGS).to(device)

eigenfun_columns = []   # one entry per loaded isometry: (top-N_TOP atom images, top-N_TOP eigenvalues, step)

# Iterate only over (classifier checkpoint, isometry) pairs the user has loaded.
loaded_steps = sorted(isometry_bundles)
loaded_paths = [ckpt_path(step) for step in loaded_steps]

for ckpt_path_, step in tqdm(
    list(zip(loaded_paths, loaded_steps)),
    desc='Computing NTK QF per checkpoint',
):
    bundle = isometry_bundles[step]

    # Load classifier weights into the shared ntk_model.
    state = torch.load(ckpt_path_, map_location=device, weights_only=False)
    ntk_model.load_state_dict(state['model_state_dict'])
    ntk_model.eval()

    iso        = bundle['isometry']
    dictionary = bundle['dictionary']
    pfk        = bundle['pushforward_kwargs']
    msk        = bundle['model_state_kwargs']
    iso.shuffle_model_state(**msk)
    iso.eval()

    indices = dictionary.get_truncated_indices(NUM_TRUNCATED_QF).to(device)

    qf = compute_ntk_qf(
        ntk_model=ntk_model, isometry=iso, dictionary=dictionary,
        indices=indices, tgt_coords=qf_coords, tgt_logabsdet=qf_logabsdet,
        pushforward_kwargs=pfk,
    )
    qf_np = qf.cpu().numpy()
    # Use argsort on the negated array to get a positive-stride descending order
    # (np.argsort(...)[::-1] returns a negative-stride view, which torch indexing rejects).
    order = np.argsort(-qf_np)[:N_TOP]
    top_indices = indices[order]
    top_eigs    = qf_np[order]

    # Render Q φ_a on the visualisation grid for the top atoms.
    with torch.no_grad():
        atoms = dictionary.get_atoms(vis_coords, top_indices)        # (N_TOP, N, C)
        _, _, pushed = iso.pushforward(
            src_coords=vis_coords,
            src_logabsdet=vis_logabsdet,
            src_field=atoms,
            **pfk,
        )
    eigenfun_columns.append({
        'step':      step,
        'images':    pushed.squeeze(-1).cpu().numpy(),   # (N_TOP, N_vis^2)
        'evals':     top_eigs,
        'top_atoms': top_indices.detach().cpu(),         # (N_TOP, d+1) — for high-res inset re-evaluation
    })


In [ ]:
# Shared colour range per checkpoint row, used by both the baseline and the
# function-space mosaic plots below.  row_vmaxes[step] = max |value| across
# all eigenfunctions shown for that step in either plot.
row_vmaxes = {}
for res in baseline_results:
    row_vmaxes[res['step']] = float(np.abs(res['evecs']).max()) + 1e-12
for col in eigenfun_columns:
    step = col['step']
    fs_vmax = float(np.abs(col['images']).max()) + 1e-12
    row_vmaxes[step] = max(row_vmaxes.get(step, 0.0), fs_vmax)
print('Shared vmax per checkpoint:')
for step, v in sorted(row_vmaxes.items()):
    print(f'  step {step:5d}:  vmax = {v:.4f}')

In [ ]:
# Final grid:  rows = loaded isometries (training-step order)
#              columns = top-N_TOP eigenfunctions (descending λ)
# Matches the layout of the direct-eigendecomposition baseline above.
# Colorbar column on the right uses the shared per-row scale from row_vmaxes.
n_rows = len(eigenfun_columns)
if n_rows == 0:
    raise RuntimeError('No isometries were loaded; nothing to plot.')

fig, axes = plt.subplots(
    n_rows, N_TOP + 1,
    figsize=(N_TOP * 1.2 + 0.4, max(n_rows, 2) * 1.2),
    gridspec_kw={'wspace': 0.02, 'hspace': 0.05,
                 'width_ratios': [1] * N_TOP + [0.05]},
    squeeze=False,
)

# Capture the vmax used per row so the inset below uses the exact same scale.
_used_vmax_fs = {}
for r, col in enumerate(eigenfun_columns):
    vmax = row_vmaxes.get(col['step'], float(np.abs(col['images']).max()) + 1e-12)
    _used_vmax_fs[col['step']] = vmax
    im = None
    for c in range(N_TOP):
        ax = axes[r, c]
        ax.set_xticks([]); ax.set_yticks([])
        img_flat = col['images'][c]
        v_img = img_flat.reshape(N_VIS_ATOM, N_VIS_ATOM).T
        im = ax.imshow(v_img, extent=(0, 1, 0, 1), origin='lower',
                  cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_aspect('equal')
        ax.text(0.03, 0.03, f'λ={col["evals"][c]:.1e}',
                transform=ax.transAxes, fontsize=5, va='bottom', ha='left',
                bbox=dict(facecolor='white', alpha=0.6, pad=1, edgecolor='none'))
        if r == 0:
            ax.set_title(f'#{c}', fontsize=8)
        if c == 0:
            ax.set_ylabel(f'step {col["step"]}', fontsize=8)
    fig.colorbar(im, cax=axes[r, N_TOP])

fig.suptitle(
    f'Top-{N_TOP} NTK eigenfunctions  (loaded {n_rows}/{NUM_CHECKPOINTS} checkpoints)\n'
    'rows = classifier checkpoint,  columns = eigenfunctions (descending λ)',
    fontsize=11,
)
plt.tight_layout(); plt.show()

# ── Inset zoom on one eigenfunction column ───────────────────────────────────
import matplotlib.patches as mpatches

inset_side = max(INSET_RIGHT - INSET_LEFT, INSET_UP - INSET_DOWN)
HIGH_RES   = max(N_VIS_ATOM, int(round(INSET_TARGET / max(inset_side, 1e-9))))
print(f'High-res Q-evaluation grid: {HIGH_RES}x{HIGH_RES}  '
      f'(box side = {inset_side:.3f}, → ≈{int(round(HIGH_RES * inset_side))}px in the inset)')

coords_hi = SquareSampler(stratified=True, add_noise=False).sample(HIGH_RES).to(device)
lad_hi    = torch.zeros(coords_hi.shape[0], device=device)

panels = []   # list of (img_hi, crop) tensors per row
for col in eigenfun_columns:
    bundle     = isometry_bundles[col['step']]
    iso        = bundle['isometry']
    dictionary = bundle['dictionary']
    pfk        = bundle['pushforward_kwargs']
    msk        = bundle['model_state_kwargs']
    iso.shuffle_model_state(**msk)
    iso.eval()
    atom_idx = col['top_atoms'][INSET_COL].to(device).unsqueeze(0)        # (1, d+1)

    with torch.no_grad():
        atoms_hi = dictionary.get_atoms(coords_hi, atom_idx)              # (1, HIGH_RES², C)
        _, _, pushed_hi = iso.pushforward(
            src_coords=coords_hi, src_logabsdet=lad_hi,
            src_field=atoms_hi, **pfk,
        )
    img_hi = pushed_hi[0].squeeze(-1).cpu().numpy().reshape(HIGH_RES, HIGH_RES).T

    x_lo = int(round(INSET_LEFT  * HIGH_RES))
    x_hi = int(round(INSET_RIGHT * HIGH_RES))
    y_lo = int(round(INSET_DOWN  * HIGH_RES))
    y_hi = int(round(INSET_UP    * HIGH_RES))
    panels.append((img_hi, img_hi[y_lo:y_hi, x_lo:x_hi]))

fig, axes = plt.subplots(
    n_rows, 2,
    figsize=(2 * 2.2, max(n_rows, 2) * 2.2),
    gridspec_kw={'wspace': 0.05, 'hspace': 0.05},
    squeeze=False,
)
for r, (col, (img_hi, crop)) in enumerate(zip(eigenfun_columns, panels)):
    vmax = _used_vmax_fs[col['step']]   # same scale as the main mosaic above

    # Left: full eigenfunction (high-res) with the inset box overlaid.
    ax_full = axes[r, 0]
    ax_full.imshow(img_hi, extent=(0, 1, 0, 1), origin='lower',
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax_full.add_patch(mpatches.Rectangle(
        (INSET_LEFT, INSET_DOWN),
        INSET_RIGHT - INSET_LEFT, INSET_UP - INSET_DOWN,
        linewidth=1.5, edgecolor='red', facecolor='none',
    ))
    ax_full.set_xticks([]); ax_full.set_yticks([]); ax_full.set_aspect('equal')
    ax_full.set_ylabel(f'step {col["step"]}', fontsize=8)
    if r == 0:
        ax_full.set_title(f'#{INSET_COL}  full', fontsize=8)

    # Right: cropped inset, framed in red.
    ax_crop = axes[r, 1]
    ax_crop.imshow(crop, extent=(INSET_LEFT, INSET_RIGHT, INSET_DOWN, INSET_UP),
                   origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    for sp in ax_crop.spines.values():
        sp.set_edgecolor('red'); sp.set_linewidth(1.5)
    ax_crop.set_xticks([]); ax_crop.set_yticks([]); ax_crop.set_aspect('equal')
    if r == 0:
        ax_crop.set_title(f'inset', fontsize=8)

fig.suptitle(
    f'Inset on eigenfunction #{INSET_COL} — function-space pushforward',
    fontsize=10,
)
plt.tight_layout(); plt.show()

## 6. Full atom mosaic — pushforwarded Fourier atoms

For each selected isometry checkpoint, push every atom in a truncated
Fourier sub-dictionary through $Q$ and tile them as a large mosaic.
This reveals which Fourier modes have been reorganised by the isometry
and in what way — complementary to the top-$K$ eigenfunction view above.

Edit `MOSAIC_STEPS` and `NUM_TRUNCATED_MOSAIC` to control which checkpoints
are shown and how large the atom dictionary is. With `NUM_TRUNCATED_MOSAIC=K`
the sub-dictionary contains all atoms with $\|k\|_\infty \le K-1$, giving
$(2K-1)^2$ atoms total.


In [ ]:
import sys
sys.path.insert(0, '..')
from notebook_helpers import plot_atom_mosaic

# ── user-editable ──────────────────────────────────────────────────────────
MOSAIC_STEPS         = sorted(ISOMETRY_CKPT_BY_STEP.keys())  # subset of loaded steps
NUM_TRUNCATED_MOSAIC = 6    # ||k||_inf <= K-1  →  (2K-1)^2 atoms per figure
N_VIS_MOSAIC         = 64  # grid resolution (N_VIS_MOSAIC^2 points)
# ───────────────────────────────────────────────────────────────────────────

from infidictionary.domain_samplers import SquareSampler as _SS

mosaic_coords    = _SS(stratified=True, add_noise=False).sample(N_VIS_MOSAIC).to(device)
mosaic_logabsdet = torch.zeros(mosaic_coords.shape[0], device=device)

for step in MOSAIC_STEPS:
    if step not in isometry_bundles:
        print(f'Step {step} not loaded; skipping.')
        continue

    bundle     = isometry_bundles[step]
    iso        = bundle['isometry']
    dictionary = bundle['dictionary']
    pfk        = bundle['pushforward_kwargs']
    msk        = bundle['model_state_kwargs']
    iso.shuffle_model_state(**msk)
    iso.eval()

    indices = dictionary.get_truncated_indices(NUM_TRUNCATED_MOSAIC).to(device)

    with torch.no_grad():
        atoms = dictionary.get_atoms(mosaic_coords, indices)   # (A, N, C)
        _, _, pushed = iso.pushforward(
            src_coords=mosaic_coords,
            src_logabsdet=mosaic_logabsdet,
            src_field=atoms,
            **pfk,
        )  # (A, N, C)

    A = indices.shape[0]
    print(f'Step {step}: {A} atoms pushed.')

    # plot_atom_mosaic(
    #     atoms.cpu(),
    #     indices.cpu(),
    #     title=rf'Fourier atoms  $\phi_{{k_1,k_2}}$   (classifier step {step})',
    #     n_vis=N_VIS_MOSAIC,
    # )
    plot_atom_mosaic(
        pushed.cpu(),
        indices.cpu(),
        title=rf'Learned atoms  $Q\phi_{{k_1,k_2}}$   (isometry at classifier step {step})',
        n_vis=N_VIS_MOSAIC,
    )
